# $\mathbb{Z}_2^F \times_\sigma \mathbb{Z}_2$ - Twisted Majorana Hamiltonians

Created: 17-08-2026

Iterate on [previous notebook](z2_f_x_z2_t_majorana_hamiltonians_z_basis.ipynb), but now use $G_F = \mathbb{Z}_2 \times_\sigma \mathbb{Z}_2$.

# Imports

In [1]:
from time import time

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from itertools import product

In [6]:
import quimb.tensor as qtn
import quimb as qu

In [7]:
from quspin.operators import hamiltonian
from quspin.operators import quantum_operator
from quspin.basis import spin_basis_1d, spinless_fermion_basis_1d, tensor_basis

In [8]:
from humanize import naturalsize

# Definitions

In [9]:
majorana_1 = [('+', 1) ,('-', 1)]
majorana_2 = [('+', 1j),  ('-', -1j)]

In [10]:
nontriv_spin_terms = [
    (1, 'x', [1,]),
    (-1, 'zxz', [0, 1, 2]),
    (1, 'y', [1,]),
    (1, 'zyz', [0,1,2])
]

In [11]:
def get_triv_terms(L, strength_scaling=1):
    terms = [
        ['x|', [[-1*strength_scaling, i] for i in range(L)]],
        ['|n', [[strength_scaling, i] for i in range(L)]],
        ['|I', [[-1*strength_scaling, i] for i in range(L)]],
    ]

    return terms

In [17]:
def get_nontriv_fermion_decoration_terms(L, strength_scaling=1):
    ss = strength_scaling

    spin_terms = list()

    for op_l, s_l in majorana_2:
        for op_r, s_r in majorana_1:
            for s_spin, spin_string, spin_sites in nontriv_spin_terms:
                strength = -1j*s_l*s_r*ss*s_spin
                base_index = [*spin_sites, 0, 1]
                all_indices = [[(x+i)%L for x in base_index] for i in range(L)]
                current_terms = [
                    f'{spin_string}|{op_l}{op_r}',
                    [[strength, *i] for i in all_indices]
                ]

                spin_terms.append(current_terms)

    fermion_terms = [
        [f'zz|I', [[-1*ss, i, (i+1)%L, i] for i in range(L)]],
        [f'zz|n', [[2*ss, i, (i+1)%L, i] for i in range(L)]]
    ]

    terms = spin_terms + fermion_terms

    return terms

In [18]:
def get_triv_to_n1_non_triv_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L, pauli=-1)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = get_triv_terms(L, 1-t)

    non_triv_terms = get_nontriv_fermion_decoration_terms(
        L,
        t
    )

    all_terms = triv_terms + non_triv_terms

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

In [19]:
L=4

In [20]:
parameters = np.linspace(0, 1, 21)

In [21]:
energies = list()

L=4
spin_basis = spin_basis_1d(L, pauli=-1)
fermion_basis = spinless_fermion_basis_1d(L)
basis = tensor_basis(spin_basis, fermion_basis)

for t in parameters:
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    energies.append(e)

/tmp/ipykernel_45549/1811742198.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [23]:
energies

[array([-8.]),
 array([-7.61357192]),
 array([-7.25699327]),
 array([-6.93468289]),
 array([-6.65249123]),
 array([-6.42112206]),
 array([-6.26574438]),
 array([-6.23971396]),
 array([-6.38064093]),
 array([-6.63746555]),
 array([-6.95170467]),
 array([-7.29706644]),
 array([-7.66250807]),
 array([-8.04252609]),
 array([-8.43385874]),
 array([-8.83431175]),
 array([-9.24228294]),
 array([-9.6565422]),
 array([-10.07611563]),
 array([-10.50021701]),
 array([-10.92820323])]

# Sweep

In [18]:
parameters = np.linspace(0, 1, 21)

## 4 site

In [19]:
L = 4

## Trivial cocycle

In [20]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_z_basis_triv_to_nontriv_n1_4_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_42085/2908583098.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 26.99it/s]


## 8 site

In [21]:
L = 8

## Trivial cocycle

In [ ]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_z_basis_triv_to_nontriv_n1_8_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_42085/2908583098.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
 67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                         | 14/21 [00:18<00:09,  1.35s/it]

## 10 site

In [ ]:
L = 10

## Trivial cocycle

In [ ]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_z_basis_triv_to_nontriv_n1_10_site_ed/{t_string}.npz', energy=e, psi=psi)